In [ ]:
# 02_merge_and_transform.ipynb
# Pipeline: Merge CSV → Pivot/Melt → Formato Final Analítico
# ============================================================================

In [8]:
# ============================================================================
# Célula 1: Imports e Configuração
# ============================================================================
import pandas as pd
import glob
import re
import os
from pathlib import Path

PASTA_PADRONIZADO = [r"C:\zardit\personal-course\data\clean\aima\padronizado", r"C:\zardit\personal-course\data\clean\sef\padronizado"]
PASTA_MERGED = r"C:\zardit\personal-course\data\clean\merged"
os.makedirs(PASTA_MERGED, exist_ok=True)

SEPARADOR = ';'
ENCODING = 'utf-8'

print("✅ Configuração carregada")
print(f"📁 Padronizados: {PASTA_PADRONIZADO}")
print(f"📁 Merged: {PASTA_MERGED}")

✅ Configuração carregada
📁 Padronizados: ['C:\\zardit\\personal-course\\data\\clean\\aima\\padronizado', 'C:\\zardit\\personal-course\\data\\clean\\sef\\padronizado']
📁 Merged: C:\zardit\personal-course\data\clean\merged


In [9]:
# ============================================================================
# Célula 2: Função para Extrair Ano do Nome do Arquivo
# ============================================================================
def extrair_ano(nome_arquivo):
    """Extrai ano de 4 dígitos do nome do arquivo"""
    match = re.search(r'(\d{4})', nome_arquivo)
    return match.group(1) if match else None

In [16]:
# ============================================================================
# Célula 3: Merge de Todos os CSVs
# ============================================================================
def merge_arquivos(pasta, padrao, nome_fonte=""):
    """
    Merge todos os CSVs que correspondem ao padrão
    """
    # Normaliza para lista
    pastas = [pasta] if isinstance(pasta, str) else pasta

    arquivos = []
    for p in pastas:
        arquivos.extend(glob.glob(os.path.join(p, padrao)))
    
    if not arquivos:
        print(f"⚠️ Nenhum arquivo encontrado para o padrão: {padrao}")
        return None
    
    print(f"\n📂 Processando {len(arquivos)} arquivos {nome_fonte}:")
    
    dfs = []
    for arquivo in sorted(arquivos):
        nome = os.path.basename(arquivo)
        ano = extrair_ano(nome)
        
        if not ano:
            print(f"   ⚠️ Não foi possível extrair ano de: {nome}")
            continue
        
        df = pd.read_csv(arquivo, sep=SEPARADOR, encoding=ENCODING)
        
        # 👇 reconverte colunas numéricas para Int64 (evita .0 no CSV final)
        numeric_cols = [c for c in df.columns if c != 'nationality']
        for col in numeric_cols:
            df[col] = df[col].astype('Int64')
        
        df.insert(0, 'year', int(ano))
        dfs.append(df)
        print(f"   📄 {nome} → ano {ano} ({len(df)} linhas)")
    
    if not dfs:
        return None
    
    df_merged = pd.concat(dfs, ignore_index=True)
    print(f"   ✅ Total: {len(df_merged)} linhas")
    return df_merged

In [17]:
# ============================================================================
# Célula 4: Merging RMA e RIFA
# ============================================================================
df_rma = merge_arquivos(PASTA_PADRONIZADO, "rma_*.csv", "RMA (AIMA)")
df_rifa = merge_arquivos(PASTA_PADRONIZADO, "rifa_*.csv", "RIFA (SEF)")
# %%
# ============================================================================
# Célula 5: Combinar e Salvar Dataset Combinado
# ============================================================================
if df_rma is not None and df_rifa is not None:
    print("\n🔄 Combinando RMA + RIFA...")
    
    # Verifica compatibilidade
    colunas_rma = set(df_rma.columns)
    colunas_rifa = set(df_rifa.columns)
    
    if colunas_rma != colunas_rifa:
        print("   ⚠️ Estruturas de colunas diferentes!")
        print(f"   RMA: {colunas_rma}")
        print(f"   RIFA: {colunas_rifa}")
        
        # Alinha colunas
        colunas_comuns = colunas_rma.intersection(colunas_rifa)
        df_rma = df_rma[[col for col in colunas_comuns if col in df_rma.columns]]
        df_rifa = df_rifa[[col for col in colunas_comuns if col in df_rifa.columns]]
        print(f"   ✅ Alinhado para colunas comuns: {colunas_comuns}")
    
    # Combina
    df_combinado = pd.concat([df_rma, df_rifa], ignore_index=True)
    df_combinado = df_combinado.sort_values(['year', 'nationality']).reset_index(drop=True)
    
    print(f"\n📊 Dataset combinado:")
    print(f"   Linhas: {len(df_combinado)}")
    print(f"   Anos: {sorted(df_combinado['year'].unique())}")
    print(f"   Colunas: {df_combinado.columns.tolist()}")
    
    # Salva
    output_combined = os.path.join(PASTA_MERGED, "residents_and_permits_by_nationality_2015_2024.csv")
    df_combinado.to_csv(output_combined, sep=SEPARADOR, encoding=ENCODING, index=False)
    print(f"\n✅ Salvo: {os.path.basename(output_combined)}")
    
else:
    print("\n❌ Não foi possível combinar os datasets!")
    # Se apenas um existir, usa ele
    df_combinado = df_rma if df_rma is not None else df_rifa


📂 Processando 2 arquivos RMA (AIMA):
   📄 rma_2023_pg36_residents_by_nationality_gender.csv → ano 2023 (192 linhas)
   📄 rma_2024_pg37_residents_by_nationality_gender.csv → ano 2024 (197 linhas)
   ✅ Total: 389 linhas

📂 Processando 8 arquivos RIFA (SEF):
   📄 rifa_2015_pg65_residents_by_nationality_gender.csv → ano 2015 (185 linhas)
   📄 rifa_2016_pg71_residents_by_nationality_gender.csv → ano 2016 (187 linhas)
   📄 rifa_2017_pg71_residents_by_nationality_gender.csv → ano 2017 (185 linhas)
   📄 rifa_2018_pg81_residents_by_nationality_gender.csv → ano 2018 (187 linhas)
   📄 rifa_2019_pg84_residents_by_nationality_gender.csv → ano 2019 (191 linhas)
   📄 rifa_2020_pg86_residents_by_nationality_gender.csv → ano 2020 (190 linhas)
   📄 rifa_2021_pg96_residents_by_nationality_gender.csv → ano 2021 (187 linhas)
   📄 rifa_2022_pg56_residents_by_nationality_gender.csv → ano 2022 (189 linhas)
   ✅ Total: 1501 linhas

🔄 Combinando RMA + RIFA...

📊 Dataset combinado:
   Linhas: 1890
   Anos: [np.

In [18]:
# ============================================================================
# Célula 6: Transformação para Formato Analítico (Melt + Pivot)
# ============================================================================
def transformar_para_analitico(df, output_path):
    """
    Transforma de formato wide (colunas por métrica+gênero) para formato analítico
    com year, nationality, gender, resident_count, permits_granted
    """
    print(f"\n🔄 Transformando para formato analítico...")
    print(f"   Dataset: {len(df)} registros")
    
    # 1. Identifica colunas de métricas
    colunas_metricas = [col for col in df.columns if col not in ['year', 'nationality']]
    print(f"   Métricas encontradas: {colunas_metricas}")
    
    # 2. Melt: transforma colunas em linhas
    df_melt = df.melt(
        id_vars=['year', 'nationality'],
        value_vars=colunas_metricas,
        var_name='metrica_genero',
        value_name='valor'
    )
    
    # 3. Separa métrica e gênero
    # Padrão: [metric]_[gender] ex: resident_count_male
    df_melt[['metrica', 'genero']] = df_melt['metrica_genero'].str.rsplit('_', n=1, expand=True)
    
    # 4. Mapeia gênero
    df_melt['genero'] = df_melt['genero'].map({
        'male': 'Masculino', 
        'female': 'Feminino',
        'masculino': 'Masculino',
        'feminino': 'Feminino'
    })
    
    # 5. Pivot para ter duas colunas de métrica
    df_final = df_melt.pivot_table(
        index=['year', 'nationality', 'genero'],
        columns='metrica',
        values='valor',
        aggfunc='sum'
    ).reset_index()
    
    # 6. Renomeia colunas
    df_final.columns.name = None
    
    # 7. Ordena e limpa
    df_final = df_final.sort_values(['year', 'nationality', 'genero']).reset_index(drop=True)
    
    # 8. Converte numéricos
    colunas_numericas = ['resident_count', 'permits_granted', 'resident', 'permits']
    for col in colunas_numericas:
        if col in df_final.columns:
            df_final[col] = pd.to_numeric(df_final[col], errors='coerce').astype('Int64')
    
    # 9. Reordena colunas (opcional)
    ordem_colunas = ['year', 'nationality', 'genero']
    outras_colunas = [col for col in df_final.columns if col not in ordem_colunas]
    df_final = df_final[ordem_colunas + outras_colunas]
    
    # 10. Salva
    df_final.to_csv(output_path, sep=SEPARADOR, encoding=ENCODING, index=False)
    print(f"\n✅ Formato analítico salvo: {os.path.basename(output_path)}")
    print(f"   {len(df_final)} registros, {len(df_final.columns)} colunas")
    
    return df_final

In [19]:
# ============================================================================
# Célula 7: Executar Transformação
# ============================================================================
if df_combinado is not None:
    output_analitico = os.path.join(PASTA_MERGED, "demographics_by_gender_clean.csv")
    df_analitico = transformar_para_analitico(df_combinado, output_analitico)


🔄 Transformando para formato analítico...
   Dataset: 1890 registros
   Métricas encontradas: ['resident_count_male', 'resident_count_female', 'permits_granted_male', 'permits_granted_female']

✅ Formato analítico salvo: demographics_by_gender_clean.csv
   3760 registros, 5 colunas


In [20]:
# ============================================================================
# Célula 8: Validação Final
# ============================================================================
print("\n" + "="*60)
print("📊 VALIDAÇÃO FINAL")
print("="*60)

if df_combinado is not None:
    print(f"\n✅ Dataset Combinado:")
    print(f"   Linhas: {len(df_combinado)}")
    print(f"   Anos: {sorted(df_combinado['year'].unique())}")
    print(f"   Colunas: {df_combinado.columns.tolist()}")

if 'df_analitico' in locals() and df_analitico is not None:
    print(f"\n✅ Dataset Analítico:")
    print(f"   Linhas: {len(df_analitico)}")
    print(f"   Colunas: {df_analitico.columns.tolist()}")
    print(f"   Gêneros: {df_analitico['genero'].unique().tolist()}")
    
    # Verifica dados
    print("\n📋 Amostra dos dados:")
    display(df_analitico.head(10))
    
    # Verifica totals
    print("\n📊 Totais por ano:")
    totals = df_analitico.groupby('year').agg({
        'resident_count': 'sum',
        'permits_granted': 'sum'
    }).round(0).astype('Int64')
    display(totals)

print("\n" + "="*60)
print("🎉 PIPELINE COMPLETO CONCLUÍDO COM SUCESSO!")
print("="*60)


📊 VALIDAÇÃO FINAL

✅ Dataset Combinado:
   Linhas: 1890
   Anos: [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
   Colunas: ['year', 'nationality', 'resident_count_male', 'resident_count_female', 'permits_granted_male', 'permits_granted_female']

✅ Dataset Analítico:
   Linhas: 3760
   Colunas: ['year', 'nationality', 'genero', 'permits_granted', 'resident_count']
   Gêneros: ['Feminino', 'Masculino']

📋 Amostra dos dados:


,year,nationality,genero,permits_granted,resident_count
0,2015,Afeganistão,Feminino,2,15
1,2015,Afeganistão,Masculino,5,26
2,2015,Albânia,Feminino,3,26
3,2015,Albânia,Masculino,2,16
4,2015,Alemanha,Feminino,499,4410
5,2015,Alemanha,Masculino,525,4625
6,2015,Andorra,Feminino,0,0
7,2015,Andorra,Masculino,0,3
8,2015,Angola,Feminino,637,9760
9,2015,Angola,Masculino,640,8487



📊 Totais por ano:


,resident_count,permits_granted
year,,
2015,388731,37851
2016,397731,46921
2017,421711,61413
2018,480300,93154
2019,590348,129155
2020,662095,118124
2021,698887,111311
2022,781915,143081
2023,1044606,328978



🎉 PIPELINE COMPLETO CONCLUÍDO COM SUCESSO!


In [23]:
# ============================================================================
# Célula 9: Relatório de Merge e Transformação por Arquivo (TXT)
# Usa variáveis já em memória (df_rma, df_rifa, df_combinado, df_analitico)
# e re-lê os ficheiros fonte para detalhar por arquivo.
# Não modifica nenhuma célula anterior.
# ============================================================================
from datetime import datetime

def _sep2(char='─', n=80): return char * n

_r2  = []
_add2 = _r2.append

_add2(_sep2('='))
_add2('RELATÓRIO DE MERGE E TRANSFORMAÇÃO — NOTEBOOK 2')
_add2(f'Data: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
_add2('Pipeline: CSVs padronizados → Merge por fonte → Merge RMA+RIFA → Melt/Pivot')
_add2(_sep2('='))

# ── Secção 1: Ficheiros fonte, por conjunto ─────────────────────────────────
_add2('')
_add2('📁 FICHEIROS FONTE (por conjunto de origem)')
_add2(_sep2())

_conjuntos = [
    ('rma_*.csv',  'RMA (AIMA)', df_rma  if 'df_rma'  in dir() else None),
    ('rifa_*.csv', 'RIFA (SEF)', df_rifa if 'df_rifa' in dir() else None),
]

for _padrao, _label, _df_merged in _conjuntos:
    _pastas_pad = [PASTA_PADRONIZADO] if isinstance(PASTA_PADRONIZADO, str) else PASTA_PADRONIZADO

    _arqs = []
    for _p in _pastas_pad:
        _arqs.extend(glob.glob(os.path.join(_p, _padrao)))
    _arqs = sorted(_arqs)
    _add2(f'\n  📂 {_label} — {len(_arqs)} ficheiro(s):')

    for _arq in _arqs:
        _nome_a   = os.path.basename(_arq)
        _df_tmp   = pd.read_csv(_arq, sep=SEPARADOR, encoding=ENCODING)
        _n_a      = len(_df_tmp)
        _cols_a   = _df_tmp.columns.tolist()
        _ano_m    = re.search(r'(\d{4})', _nome_a)
        _ano_s    = _ano_m.group(1) if _ano_m else '?'
        _add2(f'       - {_nome_a}')
        _add2(f"           Ano: {_ano_s}   Linhas: {_n_a:,}   "
              f"Colunas ({len(_cols_a)}): {_cols_a}")

        # Estatísticas numéricas por ficheiro fonte
        for _col in _df_tmp.select_dtypes(include='number').columns:
            _soma = _df_tmp[_col].sum()
            _add2(f"               {_col:<35} soma: {_soma:>12,.0f}")

    if _df_merged is not None:
        _add2(f'\n       → Após merge interno: {len(_df_merged):,} linhas totais')
        _add2(f"         Anos cobertos:        {sorted(_df_merged['year'].unique())}")
        _add2(f"         Colunas (incl. year): {_df_merged.columns.tolist()}")

# ── Secção 2: Merge RMA + RIFA ──────────────────────────────────────────────
_add2('')
_add2(_sep2())
_add2('🔄 MERGE RMA + RIFA → DATASET COMBINADO')
_add2(_sep2())

_n_rma  = len(df_rma)  if 'df_rma'  in dir() and df_rma  is not None else 0
_n_rifa = len(df_rifa) if 'df_rifa' in dir() and df_rifa is not None else 0

_add2(f"  {'Linhas RMA':<42} {_n_rma:>8,}")
_add2(f"  {'Linhas RIFA':<42} {_n_rifa:>8,}")
_add2(f"  {'Total combinado':<42} {_n_rma + _n_rifa:>8,}")

if 'df_combinado' in dir() and df_combinado is not None:
    _add2(f"  {'Anos cobertos':<42} {sorted(df_combinado['year'].unique())}")
    _add2(f"  {'Nacionalidades distintas':<42} {df_combinado['nationality'].nunique():>8,}")
    _add2(f"  {'Colunas':<42} {df_combinado.columns.tolist()}")
    _add2(f"  {'Ficheiro guardado':<42} residents_and_permits_by_nationality_2015_2024.csv")

    # Linhas por ano
    _add2("\n  Linhas por ano:")
    for _yr, _cnt in df_combinado.groupby('year').size().items():
        _add2(f"       {_yr}   {_cnt:>5,} linhas")

# ── Secção 3: Transformação Analítica ───────────────────────────────────────
_add2('')
_add2(_sep2())
_add2('📐 TRANSFORMAÇÃO ANALÍTICA (Melt + Pivot)')
_add2(_sep2())

if 'df_analitico' in dir() and df_analitico is not None:
    _cols_wide = [c for c in df_combinado.columns if c not in ['year', 'nationality']]

    _add2(f"\n  Formato WIDE (entrada — df_combinado):")
    _add2(f"       Linhas:       {len(df_combinado):,}")
    _add2(f"       Colunas:      {df_combinado.columns.tolist()}")

    _add2(f"\n  Operações aplicadas:")
    _add2(f"       1. melt()         → colunas {_cols_wide} → 'metrica_genero' + 'valor'")
    _add2("       2. rsplit('_', 1) → separa 'metrica' de 'genero'")
    _add2("       3. map()          → male→Masculino, female→Feminino")
    _add2("       4. pivot_table()  → uma coluna por métrica, linhas por género")

    _add2(f"\n  Formato LONGO (saída — df_analitico):")
    _add2(f"       Linhas:       {len(df_analitico):,}")
    _add2(f"       Colunas:      {df_analitico.columns.tolist()}")
    _add2(f"       Géneros:      {df_analitico['genero'].unique().tolist()}")
    _add2(f"       Anos:         {sorted(df_analitico['year'].unique())}")
    _add2(f"       Ficheiro:     demographics_by_gender_clean.csv")

    # Totais por ano + género
    _add2("\n  📊 Totais por ano e género:")
    _num_cols_an = [c for c in df_analitico.select_dtypes(include='number').columns
                    if c != 'year']
    _totais_an   = df_analitico.groupby(['year', 'genero'])[_num_cols_an].sum()
    _add2(_totais_an.to_string())

_add2('')
_add2(_sep2('='))
_add2('FIM DO RELATÓRIO — NOTEBOOK 2')
_add2(_sep2('='))

# ── Imprime e guarda ────────────────────────────────────────────────────────
_txt_nb2 = '\n'.join(_r2)
print(_txt_nb2)

_path_nb2 = os.path.join(PASTA_MERGED, 'relatorio_merge_nb2.txt')
with open(_path_nb2, 'w', encoding=ENCODING) as _f2:
    _f2.write(_txt_nb2)
print(f'\n💾 Relatório guardado em: {_path_nb2}')

RELATÓRIO DE MERGE E TRANSFORMAÇÃO — NOTEBOOK 2
Data: 2026-07-10 13:19:50
Pipeline: CSVs padronizados → Merge por fonte → Merge RMA+RIFA → Melt/Pivot

📁 FICHEIROS FONTE (por conjunto de origem)
────────────────────────────────────────────────────────────────────────────────

  📂 RMA (AIMA) — 2 ficheiro(s):
       - rma_2023_pg36_residents_by_nationality_gender.csv
           Ano: 2023   Linhas: 192   Colunas (5): ['nationality', 'resident_count_male', 'resident_count_female', 'permits_granted_male', 'permits_granted_female']
               resident_count_male                 soma:      553,965
               resident_count_female               soma:      490,641
               permits_granted_male                soma:      180,401
               permits_granted_female              soma:      148,577
       - rma_2024_pg37_residents_by_nationality_gender.csv
           Ano: 2024   Linhas: 197   Colunas (5): ['nationality', 'resident_count_male', 'resident_count_female', 'permits_granted